In [4]:
from config import init_env
from config import variables
import importlib
variables = importlib.reload(variables)
init_env.set_environment_variables()
import requests


### Initialize the LLM

In [5]:
from gen_ai_hub.proxy.langchain.init_models import init_llm
model = init_llm(
    'gpt-4o', 
    temperature=0,
    max_tokens=12000
)

### Getting started with a simple agent

#### Create tool

In [7]:
from langchain.tools import tool
@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

#### Create agent

In [8]:
from langchain.agents import create_agent 
agent1 = create_agent(
    model,
    tools = [search, get_weather],
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

#### Invoke agents

In [28]:

response =agent1.invoke(
    {
        "messages":[
            {
                "role": "user",
                "content": 
                """
                What is the weather in Shanghai?
                """
            }
        ]
    }
)
print(response)


{'messages': [HumanMessage(content='\n                What is the weather in Shanghai?\n                ', additional_kwargs={}, response_metadata={}, id='718cc103-2027-4ad6-81ba-13876d9334b7'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_J8ZfpUWi8mRChSbhnPGddW72', 'function': {'arguments': '{"location":"Shanghai"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 82, 'total_tokens': 97, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-CxR0xv8v1cHxVbtBFVm5MHaqwSJbe', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bb5c6-4f06-7811-a3ab-e5f7dd018a0f-0', tool_calls=[{'name': 'get_weather', 'a

### Optional: create function to display agent message

In [62]:

def invoke_agent_messages(agent, content: str):
    payload = {
        "messages": [
            {
                "role": "user", 
                "content": content
            }
        ]
    }
    response = agent.invoke(payload)
    return response["messages"]  


In [30]:
import json

def print_message_pairs(messages, verbose=False):
    """
    自动从消息序列中提取：
      - user_query：第一个 HumanMessage 的 content
      - tool_name：第一个 AIMessage.additional_kwargs.tool_calls[0].function.name
      - tool_output：第一个 ToolMessage 的 content
      - assistant_text：最后一个 AIMessage 的 content

    参数：
      - messages: 消息序列（包含 HumanMessage / AIMessage / ToolMessage 等）
      - verbose (bool): 
          True  -> 打印 JSON（包含四个键值）
          False -> 仅打印 assistant_text

    返回：
      - pairs (dict): 以上四个字段的字典，便于后续使用
    """
    # 安全提取工具名
    def extract_tool_name_from_ai(ai_msg):
        ak = getattr(ai_msg, "additional_kwargs", {})
        if isinstance(ak, dict):
            tool_calls = ak.get("tool_calls") or []
            if tool_calls:
                fn = tool_calls[0].get("function") or {}
                return fn.get("name")
        return None

    # 初始化
    user_query = None
    tool_name = None
    tool_output = None
    assistant_text = None

    # 1) 找第一个 HumanMessage 作为用户文本
    for m in messages:
        if m.__class__.__name__ == "HumanMessage":
            user_query = getattr(m, "content", None)
            break

    # 2) 找第一个 AIMessage 中的 tool_calls 取函数名
    for m in messages:
        if m.__class__.__name__ == "AIMessage":
            tool_name = extract_tool_name_from_ai(m)
            if tool_name:
                break

    # 3) 找第一个 ToolMessage 的输出
    for m in messages:
        if m.__class__.__name__ == "ToolMessage":
            tool_output = getattr(m, "content", None)
            break

    # 4) 找最后一个 AIMessage 的最终回复
    for m in reversed(messages):
        if m.__class__.__name__ == "AIMessage":
            assistant_text = getattr(m, "content", None)
            break

    pairs = {
        "user_query": user_query,
        "tool_name": tool_name,
        "tool_output": tool_output,
        #"assistant_text": assistant_text,
    }

    # 根据 verbose 控制打印
    if verbose:
        # 打印完整 JSON；ensure_ascii=False 支持中文直出（可按需移除）
        print(json.dumps(pairs, indent=2, ensure_ascii=False))
        print("\nAassistant reply:")
        print(assistant_text or "")
    else:
        # 仅打印最终助手回复；为防 None，做一下空串兜底
        print(assistant_text or "")




In [32]:
messages = invoke_agent_messages(
    agent1,
    "What is the weather in Shanghai?")
print_message_pairs(messages,verbose=True)

{
  "user_query": "What is the weather in Shanghai?",
  "tool_name": "get_weather",
  "tool_output": "Weather in Shanghai: Sunny, 72°F"
}

Aassistant reply:
The weather in Shanghai is currently sunny with a temperature of 72°F.


### Example: new agent with new tool - get online image content from its url

#### Create function

In [33]:
def get_imagedetail(image_url):
    
    # Create messages including both text and image input
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Describe the image in detail."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_url  # Use the direct image URL
                    }
                }
            ]
        }
    ]
    
    response = model.invoke(messages)
    return response.content


Optional: test the function

In [34]:
image_url="https://bmw.scene7.com/is/image/BMW/g26_bev_mp_positioning_image:16to7?fmt=webp&wid=2560&hei=1120"
response=get_imagedetail(image_url)

print(response)

The image shows a silver BMW car driving on a road with a scenic backdrop. The car is captured from the front, showcasing its sleek design and distinctive kidney grille. The headlights are on, adding to the dynamic appearance. The license plate reads "M IA 1272E." The background features a blurred view of rocky terrain and greenery, suggesting motion and speed. The sky is partly cloudy, adding to the picturesque setting.


#### Create a tool

In [35]:
from langchain.tools import tool 

@tool
def get_img(url: str) -> str:
    """Get the description of the image in the url given by user."""
    return get_imagedetail(url)

#### Create a new agent

This time add the new tool into the tool list of the new tool.

In [36]:
from langchain.agents import create_agent
from gen_ai_hub.proxy.langchain.init_models import init_llm
 
agent2 = create_agent(
    model,
    tools = [search, get_weather,get_img],
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

#### Test and compare agents

In [37]:
response = model.invoke(
    """
    Describe the image in the below url:
    "https://s.yimg.com/os/creatr-uploaded-images/2022-10/64609b80-4969-11ed-afec-2f62147a62e4"
    """
)
print(response.content)

I'm unable to view images or access external content, including URLs. However, if you describe the image to me, I can help you analyze or interpret it!


In [38]:
messages = invoke_agent_messages(
    agent1, 
    """
    Describe the image in the below url:
    "https://s.yimg.com/os/creatr-uploaded-images/2022-10/64609b80-4969-11ed-afec-2f62147a62e4"
    """
)
print_message_pairs(messages,verbose=True)

{
  "user_query": "\n    Describe the image in the below url:\n    \"https://s.yimg.com/os/creatr-uploaded-images/2022-10/64609b80-4969-11ed-afec-2f62147a62e4\"\n    ",
  "tool_name": "search",
  "tool_output": "Results for: https://s.yimg.com/os/creatr-uploaded-images/2022-10/64609b80-4969-11ed-afec-2f62147a62e4"
}

Aassistant reply:
I cannot directly view or describe images from URLs. You might want to check the URL in a web browser to see the image. If you have any other questions or need further assistance, feel free to ask!


In [39]:
messages = invoke_agent_messages(
    agent2, 
    """
    Describe the image in the below url:
    "https://s.yimg.com/os/creatr-uploaded-images/2022-10/64609b80-4969-11ed-afec-2f62147a62e4"
    """
)
print_message_pairs(messages,verbose=True)

{
  "user_query": "\n    Describe the image in the below url:\n    \"https://s.yimg.com/os/creatr-uploaded-images/2022-10/64609b80-4969-11ed-afec-2f62147a62e4\"\n    ",
  "tool_name": "get_img",
  "tool_output": "The image features a sleek, blue BMW car parked in a scenic outdoor setting. The car has a modern design with a prominent front grille and sharp, angular headlights. The BMW logo is visible on the hood. The car's body has smooth curves and a glossy finish, reflecting the surrounding environment. It is equipped with stylish alloy wheels. In the background, there is a picturesque landscape with rolling hills, a body of water, and some trees, all under a clear blue sky. The setting suggests a peaceful, natural environment."
}

Aassistant reply:
The image features a sleek, blue BMW car parked in a scenic outdoor setting. The car has a modern design with a prominent front grille and sharp, angular headlights, with the BMW logo visible on the hood. It has smooth curves and a glossy 

### Dynamic system prompt

For more advanced use cases where you need to modify the system prompt based on runtime context or agent state, you can use middleware.<br>
The <i>@dynamic_prompt</i> decorator creates middleware that generates system prompts based on the model request:

In [136]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt

agent_dyn = create_agent(
    model,
    tools = [search],
    middleware=[user_role_prompt],
    context_schema=Context
)


The system prompt will be set dynamically based on context so we get different answers to the same question.

In [138]:
query="Search for the explaination of context_schemaand and mmiddleware in Langchain Agent, and then interpret them further."

In [139]:
# When set user_role as "beginner"
response = agent_dyn.invoke(
    {"messages": [{"role": "user", "content":query}]},
    context={"user_role": "beginner"}
)
messages=response["messages"]
print_message_pairs(messages)

I wasn't able to retrieve specific information about "context_schema" and "middleware" in Langchain Agent. However, I can provide a general explanation based on typical usage in programming and AI frameworks:

1. **Context Schema**:
   - In many frameworks, a context schema refers to the structure or format of the data that is passed around within the system. It defines what kind of information is included, how it is organized, and how it can be accessed or modified. In the context of Langchain or similar AI frameworks, a context schema might specify the types of inputs and outputs that an agent can handle, including any metadata or additional parameters that are necessary for processing.

2. **Middleware**:
   - Middleware generally refers to software that acts as a bridge between different systems or layers within an application. It can be used to manage data flow, handle requests, or perform operations like logging, authentication, and error handling. In the context of Langchain Age

In [140]:
# When set user_role as "expert"
response = agent_dyn.invoke(
    {"messages": [{"role": "user", "content": query}]},
    context={"user_role": "expert"}
)
messages=response["messages"]
print_message_pairs(messages)

It seems there was an issue retrieving the specific explanations for "context_schema" and "middleware" in Langchain Agent. However, I can provide a general interpretation based on typical usage in similar contexts:

### Context Schema in Langchain Agent

**Context Schema** typically refers to a structured format or blueprint that defines the context in which an agent operates. In the context of Langchain or similar frameworks, a context schema might include:

- **Variables and Parameters**: Definitions of the variables that the agent can use or modify during its operation.
- **Data Types**: Specifications of the types of data (e.g., strings, integers, objects) that the agent can handle.
- **Constraints**: Rules or conditions that the data must satisfy.
- **Relationships**: How different pieces of data relate to each other within the context.

In Langchain, a context schema would help in structuring the input and output data for agents, ensuring that they operate within defined paramete

### MiddleWare

#### Setup: model + tools

In [3]:
from langchain_core.tools import tool
# --- Define tools ---
@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"



tools = [search, get_weather]

# --- Define model (replace your API key/config as needed) ---
from gen_ai_hub.proxy.langchain.init_models import init_llm
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=4800
)

#### Define custom Context + middleware

In [4]:

from typing import TypedDict, Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware

class Context(TypedDict):
    user_preferences: dict  # {"style": "...", "verbosity": "..."}

class CustomMiddleware(AgentMiddleware):
    # (Optional) stage-specific tool restrictions:
    # tools = [tool1, tool2]

    def before_model(self, state, runtime) -> dict[str, Any] | None:
        # Read preferences from runtime context
        prefs = runtime.context.get("user_preferences", {}) or {}
        style = str(prefs.get("style", "general")).lower()
        verbosity = str(prefs.get("verbosity", "normal")).lower()

        # Base prompt
        system_prompt = "You are a helpful assistant."

        # Style-specific guidance
        if style == "technical":
            system_prompt += " Prefer precise, technical language and include implementation details."
        elif style == "casual":
            system_prompt += " Keep explanations informal, approachable, and friendly."

        # Verbosity-specific guidance
        if verbosity in ("detailed", "high"):
            system_prompt += " Provide thorough, step-by-step explanations with concrete examples."
        elif verbosity in ("brief", "low"):
            system_prompt += " Be concise and focus on key points; use short sentences and bullet points where helpful."

        # Tune generation params (optional)
        temperature = 0.2 if style == "technical" else 0.7  # more deterministic for technical, more open for casual

        # Return updates for the upcoming model call
        return {
            "messages": [{"role": "system", "content": system_prompt}],
            "model_kwargs": {"temperature": temperature},
        }


#### Create agent

In [5]:

agent = create_agent(
    model,
    tools=tools,                       # e.g., [search, get_weather]
    middleware=[CustomMiddleware()],
    context_schema=Context,            # <-- use context, not state
    system_prompt="You are a helpful assistant. Be concise and accurate.",
)


#### Invoke the agent with user_preferences

In [6]:
query="Search for the explaination vector embeddings." 
query="Search for the story lines and theme in 三国演义 in Chinese" 
query="Search for the story lines and theme in Games of Throne and then introduce these in Chinese." 

In [9]:
# A user who prefers technical & detailed responses
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content":query
            }
        ],
    },
    context={
        "user_preferences": {
            "style": "technical",
            "verbosity": "detailed"
        }
    }
)
messages = result["messages"]

print("\n=============================== Assistant reply (technical + detailed) =================================")
print_message_pairs(messages,verbose=True)



=============================== Assistant reply (technical + detailed) =================================
{
  "user_query": "Search for the story lines and theme in Games of Throne and then introduce these in Chinese.",
  "tool_name": "search",
  "tool_output": "Results for: Game of Thrones storylines and themes"
}

Aassistant reply:
**Game of Thrones Storylines and Themes:**

**Storylines:**
1. **The Iron Throne:** The central storyline revolves around the struggle for power and control of the Iron Throne of the Seven Kingdoms. Various noble families, including the Starks, Lannisters, Baratheons, and Targaryens, vie for dominance.
   
2. **The Stark Family:** The Starks of Winterfell face numerous challenges, including betrayal, political intrigue, and the fight to reclaim their home and honor.

3. **Daenerys Targaryen's Quest:** Daenerys Targaryen's journey from exile to power, as she seeks to reclaim the throne for her family, is marked by her growth as a leader and her acquisition 

In [130]:

# A user who prefers casual & brief responses
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content":query
            }
        ],
    },
    context={
        "user_preferences": {
            "style": "casual",
            "verbosity": "brief"
        }
    }
)
messages = result["messages"]


print("\n=============================== Assistant reply (casual + brief) =================================")
print_message_pairs(messages,verbose=True)




=============================== Assistant reply (casual + brief) =================================
{
  "user_query": "Search for the story lines and theme in Games of Throne and then introduce these in Chinese.",
  "tool_name": "search",
  "tool_output": "Results for: Game of Thrones storylines and themes"
}

Aassistant reply:
**Game of Thrones Storylines and Themes:**

- **Storylines:**
  - **Power Struggles:** The series revolves around the battle for the Iron Throne among noble families.
  - **Family Dynamics:** Focuses on the relationships and conflicts within families like the Starks, Lannisters, and Targaryens.
  - **Mystical Elements:** Includes dragons, magic, and the threat of the White Walkers.
  - **Political Intrigue:** Features alliances, betrayals, and complex political maneuvers.

- **Themes:**
  - **Power and Ambition:** Explores the lengths people go to gain and maintain power.
  - **Loyalty and Betrayal:** Highlights the importance and consequences of loyalty and bet

## Multi-agent 

### 1. Subagents  

#### Example: Build a personal assistant with subagents

https://docs.langchain.com/oss/python/langchain/multi-agent/subagents-personal-assistant

In [23]:
"""
Personal Assistant Supervisor Example

This example demonstrates the tool calling pattern for multi-agent systems.
A supervisor agent coordinates specialized sub-agents (calendar and email)
that are wrapped as tools.
"""

from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# ============================================================================
# Step 1: Define low-level API tools (stubbed)
# ============================================================================

@tool
def create_calendar_event(
    title: str,
    start_time: str,  # ISO format: "2024-01-15T14:00:00"
    end_time: str,    # ISO format: "2024-01-15T15:00:00"
    attendees: list[str],  # email addresses
    location: str = ""
) -> str:
    """Create a calendar event. Requires exact ISO datetime format."""
    return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"


@tool
def send_email(
    to: list[str],      # email addresses
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send an email via email API. Requires properly formatted addresses."""
    return f"Email sent to {', '.join(to)} - Subject: {subject}"


@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,  # ISO format: "2024-01-15"
    duration_minutes: int
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    return ["09:00", "14:00", "16:00"]


# ============================================================================
# Step 2: Create specialized sub-agents
# ============================================================================

from gen_ai_hub.proxy.langchain.init_models import init_llm
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=8000
)

calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=(
        "You are a calendar scheduling assistant. "
        "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
        "into proper ISO datetime formats. "
        "Use get_available_time_slots to check availability when needed. "
        "Use create_calendar_event to schedule events. "
        "Always confirm what was scheduled in your final response."
    )
)

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=(
        "You are an email assistant. "
        "Compose professional emails based on natural language requests. "
        "Extract recipient information and craft appropriate subject lines and body text. "
        "Use send_email to send the message. "
        "Always confirm what was sent in your final response."
    )
)

# ============================================================================
# Step 3: Wrap sub-agents as tools for the supervisor
# ============================================================================

@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text


@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text


# ============================================================================
# Step 4: Create the supervisor agent
# ============================================================================

supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=(
        "You are a helpful personal assistant. "
        "You can schedule calendar events and send emails. "
        "Break down user requests into appropriate tool calls and coordinate the results. "
        "When a request involves multiple actions, use multiple tools in sequence."
    )
)

# ============================================================================
# Step 5: Use the supervisor
# ============================================================================

if __name__ == "__main__":
    # Example: User request requiring both calendar and email coordination
    user_request = (
        "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
        "and send them an email reminder about reviewing the new mockups."
    )

    print("User Request:", user_request)
    print("\n" + "="*80 + "\n")

    for step in supervisor_agent.stream(
        {"messages": [{"role": "user", "content": user_request}]}
    ):
        for update in step.values():
            for message in update.get("messages", []):
                message.pretty_print()

User Request: Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, and send them an email reminder about reviewing the new mockups.


================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_4lDsMPSYlNlQwGoIrjnR4AsG)
 Call ID: call_4lDsMPSYlNlQwGoIrjnR4AsG
  Args:
    request: Schedule a meeting with the design team next Tuesday at 2pm for 1 hour
  manage_email (call_ssoTL3JIW2jnCuIUE5a17KK7)
 Call ID: call_ssoTL3JIW2jnCuIUE5a17KK7
  Args:
    request: Send the design team an email reminder about reviewing the new mockups
================================= Tool Message =================================
Name: manage_email

I have sent an email to the design team reminding them to review the new mockups.
================================= Tool Message =================================
Name: schedule_event

The meeting with the design team has been scheduled for next Tuesday, November 7th, from 2:00 PM to 3:00 PM.

#### Build an own agent

##### New functions for tools

###### Function: Get text from website

In [ ]:

from openai import OpenAI
import requests
from bs4 import BeautifulSoup

def extract_article_text(url: str) -> str:
    """Fetch and extract main text from a webpage."""
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Remove scripts/styles
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    # Try to get main content
    article_tags = soup.find_all(["article", "p"])
    text = " ".join(tag.get_text(strip=True) for tag in article_tags)
    return text[:8000]  # Limit to ~8k chars for token safety


Test the function

In [62]:
article_url="https://www.duh.de/presse/pressemitteilungen/pressemitteilung/hvo100-noch-schmutziger-als-herkoemmlicher-diesel-abgasmessungen-der-deutschen-umwelthilfe-zerstoeren/"
 
response=extract_article_text(article_url).strip()
print(response)


Die Deutsche Umwelthilfe im Netz Pressemitteilung •	Mehr ultrafeine Feinstaub-Partikel und Stickoxide: DUH-Abgasmessung im realen Straßenbetrieb an Euro-5-Diesel-Pkw widerlegt Mythos von „besonders nachhaltigem“ Dieselkraftstoff HVO100 •	DUH fordert Verkehrsminister Wissing auf, seine Behauptungen zu unterlassen, dass mit HVO100 „lokale Umweltbelastung in Städten und Kommunen“ reduziert werde, der Minister muss ihm vorliegende Untersuchungen über erhöhte Stickoxid-Emissionen zum neuen Dieselkraftstoff veröffentlichen •	DUH fordert Nachrüstung schmutziger Diesel-Pkw und Nutzfahrzeuge auf Kosten der Hersteller statt klima- und gesundheitsschädlicher Pseudo-Alternativen Berlin, 27.6.2024: Der neue Dieselkraftstoff HVO100 soll die klimaschädlichen Treibhausgasemissionen um „bis zu 90 Prozent“ verringern und gleichzeitig die „lokale Umweltbelastung in Städten und Kommunen“ reduzieren – so wirbt Bundesverkehrsminister Wissing für diesen angeblichen Wunderkraftstoff. Messungen der Deutschen U

###### Function: Translate text into target language

In [5]:
def text_translation(target_language: str = "Chinese", text: str="") -> str:
     
    system = (
        f"You are a professional translator. Precisely translate into {target_language}. "
        "Return ONLY the accurate translated text. No explanations, no quotes."
    )
    
    user = f"Target language: {target_language}\n\nText:\n{text}"

    messages=[
        ("system", system),
        ("user",user),
    ]

    # print(messages)
    response = model.invoke(messages)
    return response.content 

Test the function

In [65]:
#The article is written in German
article_url="https://www.duh.de/presse/pressemitteilungen/pressemitteilung/hvo100-noch-schmutziger-als-herkoemmlicher-diesel-abgasmessungen-der-deutschen-umwelthilfe-zerstoeren/"
sourcetext=extract_article_text(article_url).strip()

#In this function the default target languague is Chinese 
print(text_translation(text=sourcetext))

德国环境援助网络新闻稿

• 更多超细颗粒物和氮氧化物：DUH在实际道路运行中对欧5柴油车的排放测量推翻了“特别可持续”柴油燃料HVO100的神话
• DUH要求交通部长Wissing停止声称HVO100可以减少“城市和社区的局部环境污染”，部长必须公布他所掌握的关于新柴油燃料氮氧化物排放增加的研究
• DUH要求制造商承担肮脏柴油车和商用车的改装费用，而不是使用对气候和健康有害的伪替代品

柏林，2024年6月27日：新柴油燃料HVO100据称可以减少“高达90%”的温室气体排放，同时减少“城市和社区的局部环境污染”——联邦交通部长Wissing如此宣传这一所谓的神奇燃料。然而，德国环境援助（DUH）对一辆欧5柴油车的测量显示：新柴油燃料HVO100比传统柴油更有害健康。DUH自有的排放控制研究所（EKI）的测量显示，与传统柴油相比，使用HVO100时，柴油废气毒物NOx的排放增加了20%。ADAC的测量显示，特别有害健康的超细颗粒物数量显著增加。HVO100是一种伪解决方案，其燃烧和生产往往伴随着对气候和生物多样性的严重副作用。使用HVO100和其他“替代”燃料不是根本性交通转型和现有肮脏柴油车改装的替代方案。

DUH排放控制研究所所长Axel Friedrich：“我们对一辆大众途锐欧5的测量显示，HVO100的氮氧化物排放比传统柴油高出约20%。特别令人担忧的是，超细颗粒物也在增加。这些颗粒物对健康特别有害，因为它们可以深入到体内直至血管。HVO燃料也被不合理地排除在CO2定价之外。这必须立即停止。”

HVO100据称仅由旧煎炸油和其他残留物制成，但实际上显然也由专门种植的植物油如棕榈油制成。原材料的有限可用性、掺杂其他有价值原材料的欺诈行为以及用于种植棕榈油、大豆等的巨大土地占用，导致HVO的使用对气候和生物多样性产生部分严重影响。旧煎炸油和含油的残留和废弃物的供应量不足，并且已经在化学工业中作为有价值的原材料使用。在使用旧煎炸油生产HVO100时，工业必须通过原油产品来替代缺失的数量。在诚实的整体考量中，实际的温室气体排放因此往往甚至高于传统柴油燃料。

DUH联邦执行董事Jürgen Resch：“我们要求联邦交通部长Volker Wissing立即停止关于HVO100柴油在城市和社区中减少环境污染的错误说法。我们还想知道他从何时起就知道这些对健康

In [66]:
#Test with input target languague
target_language="English"

print(text_translation(target_language="English",text=sourcetext))

The German Environmental Aid on the Net Press Release • More ultrafine particulate matter and nitrogen oxides: DUH emissions measurement in real road operation on Euro-5 diesel cars disproves the myth of "particularly sustainable" diesel fuel HVO100 • DUH calls on Transport Minister Wissing to refrain from claiming that HVO100 reduces "local environmental pollution in cities and municipalities," the minister must publish studies available to him on increased nitrogen oxide emissions from the new diesel fuel • DUH demands retrofitting of dirty diesel cars and commercial vehicles at the manufacturers' expense instead of climate- and health-damaging pseudo-alternatives Berlin, 27.6.2024: The new diesel fuel HVO100 is supposed to reduce climate-damaging greenhouse gas emissions by "up to 90 percent" while also reducing "local environmental pollution in cities and municipalities" – this is how Federal Transport Minister Wissing promotes this alleged miracle fuel. However, measurements by th

###### Function: Get description of a Web imagme

In [6]:
def image_detail(image_url):
    
    # Create messages including both text and image input
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Describe the image in detail in Chinese."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_url  # Use the direct image URL
                    }
                }
            ]
        }
    ]
    
    response = model.invoke(messages)
    return response.content


Test the function

In [9]:
image_url="https://bmw.scene7.com/is/image/BMW/g26_bev_mp_positioning_image:16to7?fmt=webp&wid=2560&hei=1120"
 
response=image_detail(image_url)

print(response)

这张图片展示了一辆银色的宝马汽车，正行驶在一条公路上。汽车的前脸设计具有现代感，标志性的双肾型进气格栅位于中央，车牌号为“M IA 1272E”。两侧的前大灯设计犀利，呈现出一种动感的视觉效果。背景中可以看到模糊的山脉和蓝天，显示出车辆正在快速行驶。整体画面给人一种速度与力量的感觉。


##### Define new function as tools

In [14]:
from langchain.tools import tool
from openai import OpenAI
import requests
from bs4 import BeautifulSoup


@tool
def get_web_article(
    article_url: str
) -> str:
    """
    Get the article content from the website url given by user.
    """
    response = requests.get(article_url, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Remove scripts/styles
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    # Try to get main content
    article_tags = soup.find_all(["article", "p"])
    text = " ".join(tag.get_text(strip=True) for tag in article_tags)
    return text[:10000]  # Limit to ~8k chars for token safety



@tool
def get_translation(
    target_language: str = "Chinese", 
    text: str=""
) -> str:
    """
    Translate given text into `target_language`.
    By default the value of `target_language` is 'Chinese'.
    """    
    system = (
        f"You are a professional translator. Precisely translate into {target_language}. "
        "Return ONLY the accurate translated text. No explanations, no quotes."
    )
    user = f"Target language: {target_language}\n\nText:\n{text}"
    messages=[
        ("system", system),
        ("user",user),
    ]
    response = model.invoke(messages)
    return response.content 


@tool
def get_web_image(
    target_language: str = "Chinese", 
    image_url: str=""
) -> str:
    """
    Get the image content from the website url given by user.
    Describe the image in detail in user expected target languague.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"Describe the image in detail in {target_language}."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_url  # Use the direct image URL
                    }
                }
            ]
        }
    ]
    response = model.invoke(messages)
    return response.content


##### Create specialized sub-agents


###### Creat the web article sub-agent

The agent understands fetch content from the input url and then translate it into target languague(by default Chinese).  

In [ ]:
from langchain.agents import create_agent

WEB_ARTICLE_AGENT_PROMPT = (
    "You are a helpful Ai assitant."
    "Help user to get accurate information from the given website url"
    "Use the tool get_web_article to get the article content, do not remove any part, do not do any summary or ellabaration,do not merge the text into one paragragh."
    "Then use the tool get_translation to get the translated version of the content, the target languague is Chinese by default,do not remove any part, do not do any summary or ellabaration,do not merge the text into one paragragh."

)

web_article_subagent = create_agent(
    model,
    tools=[get_web_article, get_translation],
    system_prompt=WEB_ARTICLE_AGENT_PROMPT,
)

Test the agent to see how it calls the 2 tools

In [19]:
article_url="https://www.duh.de/presse/pressemitteilungen/pressemitteilung/hvo100-noch-schmutziger-als-herkoemmlicher-diesel-abgasmessungen-der-deutschen-umwelthilfe-zerstoeren/"

query = article_url

for step in web_article_subagent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_web_article (call_P3ufin55fKIPwoUDQk7OTYU9)
 Call ID: call_P3ufin55fKIPwoUDQk7OTYU9
  Args:
    article_url: https://www.duh.de/presse/pressemitteilungen/pressemitteilung/hvo100-noch-schmutziger-als-herkoemmlicher-diesel-abgasmessungen-der-deutschen-umwelthilfe-zerstoeren/
================================= Tool Message =================================
Name: get_web_article

Die Deutsche Umwelthilfe im Netz Pressemitteilung •	Mehr ultrafeine Feinstaub-Partikel und Stickoxide: DUH-Abgasmessung im realen Straßenbetrieb an Euro-5-Diesel-Pkw widerlegt Mythos von „besonders nachhaltigem“ Dieselkraftstoff HVO100 •	DUH fordert Verkehrsminister Wissing auf, seine Behauptungen zu unterlassen, dass mit HVO100 „lokale Umweltbelastung in Städten und Kommunen“ reduziert werde, der Minister muss ihm vorliegende Untersuchungen über erhöhte Stickoxid-Emissionen zum neuen Dieselkraftstoff veröffentlichen 

###### Creat the web image sub-agent

The email agent handles message composition and sending. It focuses on extracting recipient information, crafting appropriate subject lines and body text, and managing email communication.

In [35]:
WEB_IMAGE_PROMPT = (
    "You are a helpful Ai assitant."
    "Help user to get accurate information from the given website url"
    "Use the tool get_web_image to get the image information"
    "Describe the image in in user expected target languague,the target languague is Chinese by default." 
)

web_image_subagent = create_agent(
    model,
    tools=[get_web_image],
    system_prompt=WEB_IMAGE_PROMPT,
)

Test the sub-agent 

In [38]:
image_url="https://bmw.scene7.com/is/image/BMW/g26_bev_mp_positioning_image:16to7?fmt=webp&wid=2560&hei=1120"+"English"
query = image_url

for step in web_image_subagent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_web_image (call_hml4ovurhs1YK8nEfqJWjPiG)
 Call ID: call_hml4ovurhs1YK8nEfqJWjPiG
  Args:
    target_language: English
    image_url: https://bmw.scene7.com/is/image/BMW/g26_bev_mp_positioning_image:16to7?fmt=webp&wid=2560&hei=1120
================================= Tool Message =================================
Name: get_web_image

The image shows a silver BMW car driving on a road. The car is captured from the front, showcasing its sleek design and distinctive kidney grille. The headlights are on, and the license plate reads "M IA 1272 E." The background features a blurred view of rocky terrain and a clear blue sky, suggesting the car is moving at a high speed. The overall scene conveys a sense of motion and performance.
================================== Ai Message ==================================

The image shows a silver BMW car driving on a road. The car is captured from the front,

##### Create the supervisor agent

###### Wrap sub-agents as tools

Now wrap each sub-agent as a tool that the supervisor can invoke. This is the key architectural step that creates the layered system. The supervisor will see high-level tools like "web_article_event", not low-level tools like "get_web_article".<p>
The tool descriptions help the supervisor decide when to use each tool, so make them clear and specific. We return only the sub-agent’s final response, as the supervisor doesn’t need to see intermediate reasoning or tool calls.

In [39]:
@tool
def web_article_event(request: str) -> str:
    """
    Web article events
    Use this when the user gives an url for an article
    Help user to get accurate information from the given website url"
    """
    result = web_article_subagent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].content


@tool
def web_image_event(request: str) -> str:
    """
    Web image events
    Use this when the user gives an url for an image
    Help user to get accurate detail description of the image from the given website url"
    """
    result = web_image_subagent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].content



###### Create the supervisor agent with high-level tools

Now create the supervisor that orchestrates the sub-agents. The supervisor only sees high-level tools and makes routing decisions at the domain level, not the individual API level.

In [40]:
SUPERVISOR_PROMPT = (
    """
    You are a helpful AI assistant.
    You can help users get detailed information when given a website URL.
    Choose exactly one tool based on whether the URL is an image or an article.
    When a tool is called, return ONLY the tool's output as the final answer as AIMessage — do not add extra commentary.
    """
)

supervisor_agent = create_agent(
    model,
    tools=[web_article_event, web_image_event],
    system_prompt=SUPERVISOR_PROMPT,
)



##### Test the supervisor agent

In [41]:
article_url="https://www.duh.de/presse/pressemitteilungen/pressemitteilung/hvo100-noch-schmutziger-als-herkoemmlicher-diesel-abgasmessungen-der-deutschen-umwelthilfe-zerstoeren/"
image_url= "https://bmw.scene7.com/is/image/BMW/g26_bev_mp_positioning_image:16to7?fmt=webp&wid=2560&hei=1120" 

###### Test case 1: Send an URL of article

In [42]:
query =article_url

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()


================================== Ai Message ==================================
Tool Calls:
  web_article_event (call_l2Ik1aBKFAjlTGKUR4ZifMHX)
 Call ID: call_l2Ik1aBKFAjlTGKUR4ZifMHX
  Args:
    request: https://www.duh.de/presse/pressemitteilungen/pressemitteilung/hvo100-noch-schmutziger-als-herkoemmlicher-diesel-abgasmessungen-der-deutschen-umwelthilfe-zerstoeren/
================================= Tool Message =================================
Name: web_article_event

德国环境援助组织网络新闻稿 • 更多超细颗粒物和氮氧化物：DUH在实际道路运行中对欧5柴油车的排放测量推翻了“特别可持续”柴油燃料HVO100的神话 • DUH要求交通部长Wissing停止声称使用HVO100可以“减少城市和社区的局部环境污染”，部长必须公布他所掌握的关于新柴油燃料氮氧化物排放增加的研究 • DUH要求制造商承担费用对污染柴油车和商用车进行改装，而不是使用对气候和健康有害的伪替代品 柏林，2024年6月27日：新柴油燃料HVO100据称可以减少“高达90%”的温室气体排放，同时减少“城市和社区的局部环境污染”——联邦交通部长Wissing如此宣传这一所谓的神奇燃料。然而，德国环境援助组织（DUH）对一辆欧5柴油车的测量显示：新柴油燃料HVO100比传统柴油更具健康危害。DUH自有的排放控制研究所（EKI）的测量显示，与传统柴油相比，使用HVO100时柴油废气毒物NOx的排放增加了20%。ADAC的测量显示，特别有害健康的超细颗粒物数量显著增加。HVO100是一种虚假解决方案，其燃烧和生产常伴随对气候和生物多样性严重的副作用。使用HVO100和其他“替代”燃料不是对根本性交通转型和对现有污染柴油车进行改装的替代方案

###### TEst case 2: Send an URL of image

In [43]:
query =image_url

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()


================================== Ai Message ==================================
Tool Calls:
  web_image_event (call_OwrLE77LU03nO5pCK83APE20)
 Call ID: call_OwrLE77LU03nO5pCK83APE20
  Args:
    request: https://bmw.scene7.com/is/image/BMW/g26_bev_mp_positioning_image:16to7?fmt=webp&wid=2560&hei=1120
================================= Tool Message =================================
Name: web_image_event

这张图片展示了一辆银色的宝马汽车，正行驶在一条公路上。汽车的前脸设计具有现代感，标志性的双肾型进气格栅位于中央，车牌号为“M IA 1272E”。两侧的前大灯设计犀利，呈现出一种动感的视觉效果。背景中可以看到模糊的山脉和蓝天，显示出车辆正在快速行驶。整体画面给人一种速度与力量的感觉。
================================== Ai Message ==================================

这张图片展示了一辆银色的宝马汽车，正行驶在一条公路上。汽车的前脸设计具有现代感，标志性的双肾型进气格栅位于中央，车牌号为“M IA 1272E”。两侧的前大灯设计犀利，呈现出一种动感的视觉效果。背景中可以看到模糊的山脉和蓝天，显示出车辆正在快速行驶。整体画面给人一种速度与力量的感觉。


##### Create the final simple function

In [44]:
def weburl_detail(query):
    response=supervisor_agent.invoke(
        {
            "messages":[
                {
                    "role": "user",
                    "content": query
                }
            ]
        }
    )
    AIMessage=response['messages'][-1]
    print(AIMessage.content)

In [ ]:
query =article_url
weburl_detail(query)c

德国环境援助组织网络新闻稿

• 更多超细颗粒物和氮氧化物：DUH在实际道路运行中对欧5柴油车的排放测量推翻了“特别可持续”柴油燃料HVO100的神话
• DUH要求交通部长Wissing停止声称HVO100可以“减少城市和社区的局部环境负担”，部长必须公布他所掌握的关于新柴油燃料氮氧化物排放增加的研究
• DUH要求制造商承担费用对污染柴油车和商用车进行改装，而不是使用对气候和健康有害的伪替代品

柏林，2024年6月27日：新柴油燃料HVO100据称可以减少“高达90%”的温室气体排放，同时减少“城市和社区的局部环境负担”——联邦交通部长Wissing如此宣传这一所谓的神奇燃料。然而，德国环境援助组织（DUH）对一辆欧5柴油车的测量显示：新柴油燃料HVO100比传统柴油更具健康危害。DUH自有的排放控制研究所（EKI）的测量显示，与传统柴油相比，使用HVO100时柴油废气毒物NOx的排放增加了20%。ADAC的测量显示，特别有害健康的超细颗粒物数量显著增加。HVO100是一种虚假解决方案，其燃烧和生产过程中常伴有对气候和生物多样性严重的副作用。使用HVO100和其他“替代”燃料不是对根本性交通转型和对现有污染柴油车进行改装的替代方案。

DUH排放控制研究所所长Axel Friedrich：“我们对一辆大众途锐欧5的测量显示，使用HVO100时氮氧化物排放比传统柴油高约20%。特别令人担忧的是，超细颗粒物也在增加。这些颗粒物对健康特别有害，因为它们可以深入身体直至血液。HVO燃料也被不合理地排除在CO2定价之外。这必须立即停止。”

HVO100据称仅由旧煎炸油和其他残留物制成，但实际上显然也由专门种植的植物油如棕榈油制成。原材料的有限供应、掺入其他有价值原材料的欺诈行为以及用于种植棕榈油、大豆等的巨大土地占用导致HVO的使用伴随对气候和生物多样性严重的影响。旧煎炸油和含油的残留和废弃物供应不足，并且已经作为化学工业中的有价值原材料使用。在使用旧煎炸油生产HVO100时，工业必须通过原油产品来替代缺失的数量。在诚实的整体评估中，实际的温室气体排放因此往往甚至高于传统柴油燃料。

DUH联邦执行董事Jürgen Resch：“我们要求联邦交通部长Volker Wissing立即停止关于HVO100柴油在城市和社区中减少环境负担的错误说法。我们还想知道他从何时开始知

In [46]:
query =image_url+"English"
weburl_detail(query)

The image shows a silver BMW car driving on a road. The car is captured from the front, showcasing its sleek design and distinctive kidney grille. The headlights are on, adding to the dynamic appearance. The background features a blurred view of rocky terrain and a clear blue sky, suggesting the car is moving at a high speed. The license plate is visible, and the overall scene conveys a sense of motion and performance.


### 2. Handoffs 

#### Example: build customer support with handoffs

https://docs.langchain.com/oss/python/langchain/multi-agent/handoffs-customer-support

In [49]:
"""
Customer Support State Machine Example

This example demonstrates the state machine pattern.
A single agent dynamically changes its behavior based on the current_step state,
creating a state machine for sequential information collection.
"""

import uuid

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from typing import Callable, Literal
from typing_extensions import NotRequired

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, SummarizationMiddleware
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import tool, ToolRuntime

from gen_ai_hub.proxy.langchain.init_models import init_llm
model = init_llm(
    'gpt-4o', 
    temperature=0,
    max_tokens=12000
)


# Define the possible workflow steps
SupportStep = Literal["warranty_collector", "issue_classifier", "resolution_specialist"]


class SupportState(AgentState):
    """State for customer support workflow."""

    current_step: NotRequired[SupportStep]
    warranty_status: NotRequired[Literal["in_warranty", "out_of_warranty"]]
    issue_type: NotRequired[Literal["hardware", "software"]]


@tool
def record_warranty_status(
    status: Literal["in_warranty", "out_of_warranty"],
    runtime: ToolRuntime[None, SupportState],
) -> Command:
    """Record the customer's warranty status and transition to issue classification."""
    return Command(
        update={
            "messages": [
                ToolMessage(
                    content=f"Warranty status recorded as: {status}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
            "warranty_status": status,
            "current_step": "issue_classifier",
        }
    )


@tool
def record_issue_type(
    issue_type: Literal["hardware", "software"],
    runtime: ToolRuntime[None, SupportState],
) -> Command:
    """Record the type of issue and transition to resolution specialist."""
    return Command(
        update={
            "messages": [
                ToolMessage(
                    content=f"Issue type recorded as: {issue_type}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
            "issue_type": issue_type,
            "current_step": "resolution_specialist",
        }
    )


@tool
def escalate_to_human(reason: str) -> str:
    """Escalate the case to a human support specialist."""
    # In a real system, this would create a ticket, notify staff, etc.
    return f"Escalating to human support. Reason: {reason}"


@tool
def provide_solution(solution: str) -> str:
    """Provide a solution to the customer's issue."""
    return f"Solution provided: {solution}"


# Define prompts as constants
WARRANTY_COLLECTOR_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STEP: Warranty verification

At this step, you need to:
1. Greet the customer warmly
2. Ask if their device is under warranty
3. Use record_warranty_status to record their response and move to the next step

Be conversational and friendly. Don't ask multiple questions at once."""

ISSUE_CLASSIFIER_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STEP: Issue classification
CUSTOMER INFO: Warranty status is {warranty_status}

At this step, you need to:
1. Ask the customer to describe their issue
2. Determine if it's a hardware issue (physical damage, broken parts) or software issue (app crashes, performance)
3. Use record_issue_type to record the classification and move to the next step

If unclear, ask clarifying questions before classifying."""

RESOLUTION_SPECIALIST_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STEP: Resolution
CUSTOMER INFO: Warranty status is {warranty_status}, issue type is {issue_type}

At this step, you need to:
1. For SOFTWARE issues: provide troubleshooting steps using provide_solution
2. For HARDWARE issues:
   - If IN WARRANTY: explain warranty repair process using provide_solution
   - If OUT OF WARRANTY: escalate_to_human for paid repair options

Be specific and helpful in your solutions."""


# Step configuration: maps step name to (prompt, tools, required_state)
STEP_CONFIG = {
    "warranty_collector": {
        "prompt": WARRANTY_COLLECTOR_PROMPT,
        "tools": [record_warranty_status],
        "requires": [],
    },
    "issue_classifier": {
        "prompt": ISSUE_CLASSIFIER_PROMPT,
        "tools": [record_issue_type],
        "requires": ["warranty_status"],
    },
    "resolution_specialist": {
        "prompt": RESOLUTION_SPECIALIST_PROMPT,
        "tools": [provide_solution, escalate_to_human],
        "requires": ["warranty_status", "issue_type"],
    },
}


@wrap_model_call
def apply_step_config(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """Configure agent behavior based on the current step."""
    # Get current step (defaults to warranty_collector for first interaction)
    current_step = request.state.get("current_step", "warranty_collector")

    # Look up step configuration
    step_config = STEP_CONFIG[current_step]

    # Validate required state exists
    for key in step_config["requires"]:
        if request.state.get(key) is None:
            raise ValueError(f"{key} must be set before reaching {current_step}")

    # Format prompt with state values
    system_prompt = step_config["prompt"].format(**request.state)

    # Inject system prompt and step-specific tools
    request = request.override(
        system_prompt=system_prompt,
        tools=step_config["tools"],
    )

    return handler(request)


# Collect all tools from all step configurations
all_tools = [
    record_warranty_status,
    record_issue_type,
    provide_solution,
    escalate_to_human,
]

# Create the agent with step-based configuration and summarization
agent = create_agent(
    model,
    tools=all_tools,
    state_schema=SupportState,
    middleware=[
        apply_step_config,
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 4000),
            keep=("messages", 10)
        )
    ],
    checkpointer=InMemorySaver(),
)


# ============================================================================
# Test the workflow
# ============================================================================

if __name__ == "__main__":
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    result = agent.invoke(
        {"messages": [HumanMessage("Hi, my phone screen is cracked")]},
        config
    )

    result = agent.invoke(
        {"messages": [HumanMessage("Yes, it's still under warranty")]},
        config
    )

    result = agent.invoke(
        {"messages": [HumanMessage("The screen is physically cracked from dropping it")]},
        config
    )

    result = agent.invoke(
        {"messages": [HumanMessage("What should I do?")]},
        config
    )
    for msg in result['messages']:
        msg.pretty_print()

================================ Human Message =================================

Hi, my phone screen is cracked
================================== Ai Message ==================================

Hi there! I'm sorry to hear about your cracked phone screen. Let's see if we can help you out. Is your device currently under warranty?
================================ Human Message =================================

Yes, it's still under warranty
================================== Ai Message ==================================
Tool Calls:
  record_warranty_status (call_cItp5zhGUhZKB5267PBqhutw)
 Call ID: call_cItp5zhGUhZKB5267PBqhutw
  Args:
    status: in_warranty
================================= Tool Message =================================
Name: record_warranty_status

Warranty status recorded as: in_warranty
================================== Ai Message ==================================
Tool Calls:
  record_issue_type (call_Bi6tkILB36U1EajtycNZEHF8)
 Call ID: call_Bi6tkILB36U1EajtycNZEH

#### Build an own agent

##### Define custom state

In [ ]:
from langchain.agents import AgentState
from typing_extensions import NotRequired
from typing import Literal

# Define the possible workflow steps
SupportStep = Literal["warranty_collector", "issue_classifier", "resolution_specialist"]  

class SupportState(AgentState):  
    """State for customer support workflow."""
    current_step: NotRequired[SupportStep]  
    warranty_status: NotRequired[Literal["in_warranty", "out_of_warranty"]]
    issue_type: NotRequired[Literal["hardware", "software"]]

##### Create tools that manage workflow state

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command

@tool
def record_warranty_status(
    status: Literal["in_warranty", "out_of_warranty"],
    runtime: ToolRuntime[None, SupportState],
) -> Command:  
    """Record the customer's warranty status and transition to issue classification."""
    return Command(  
        update={  
            "messages": [
                ToolMessage(
                    content=f"Warranty status recorded as: {status}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
            "warranty_status": status,
            "current_step": "issue_classifier",  
        }
    )


@tool
def record_issue_type(
    issue_type: Literal["hardware", "software"],
    runtime: ToolRuntime[None, SupportState],
) -> Command:  
    """Record the type of issue and transition to resolution specialist."""
    return Command(  
        update={  
            "messages": [
                ToolMessage(
                    content=f"Issue type recorded as: {issue_type}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
            "issue_type": issue_type,
            "current_step": "resolution_specialist",  
        }
    )


@tool
def escalate_to_human(reason: str) -> str:
    """Escalate the case to a human support specialist."""
    # In a real system, this would create a ticket, notify staff, etc.
    return f"Escalating to human support. Reason: {reason}"


@tool
def provide_solution(solution: str) -> str:
    """Provide a solution to the customer's issue."""
    return f"Solution provided: {solution}"

In [54]:
"""
Multi-Source Knowledge Router Example

This example demonstrates the router pattern for multi-agent systems.
A router classifies queries, routes them to specialized agents in parallel,
and synthesizes results into a combined response.
"""

import operator
from typing import Annotated, Literal, TypedDict

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from pydantic import BaseModel, Field


# State definitions
class AgentInput(TypedDict):
    """Simple input state for each subagent."""
    query: str


class AgentOutput(TypedDict):
    """Output from each subagent."""
    source: str
    result: str


class Classification(TypedDict):
    """A single routing decision: which agent to call with what query."""
    source: Literal["github", "notion", "slack"]
    query: str


class RouterState(TypedDict):
    query: str
    classifications: list[Classification]
    results: Annotated[list[AgentOutput], operator.add]
    final_answer: str


# Structured output schema for classifier
class ClassificationResult(BaseModel):
    """Result of classifying a user query into agent-specific sub-questions."""
    classifications: list[Classification] = Field(
        description="List of agents to invoke with their targeted sub-questions"
    )


# Tools
@tool
def search_code(query: str, repo: str = "main") -> str:
    """Search code in GitHub repositories."""
    return f"Found code matching '{query}' in {repo}: authentication middleware in src/auth.py"


@tool
def search_issues(query: str) -> str:
    """Search GitHub issues and pull requests."""
    return f"Found 3 issues matching '{query}': #142 (API auth docs), #89 (OAuth flow), #203 (token refresh)"


@tool
def search_prs(query: str) -> str:
    """Search pull requests for implementation details."""
    return f"PR #156 added JWT authentication, PR #178 updated OAuth scopes"


@tool
def search_notion(query: str) -> str:
    """Search Notion workspace for documentation."""
    return f"Found documentation: 'API Authentication Guide' - covers OAuth2 flow, API keys, and JWT tokens"


@tool
def get_page(page_id: str) -> str:
    """Get a specific Notion page by ID."""
    return f"Page content: Step-by-step authentication setup instructions"


@tool
def search_slack(query: str) -> str:
    """Search Slack messages and threads."""
    return f"Found discussion in #engineering: 'Use Bearer tokens for API auth, see docs for refresh flow'"


@tool
def get_thread(thread_id: str) -> str:
    """Get a specific Slack thread."""
    return f"Thread discusses best practices for API key rotation"



from gen_ai_hub.proxy.langchain.init_models import init_llm
model = init_llm(
    'gpt-4o', 
    temperature=0,
    max_tokens=12000
)
router_llm = model


github_agent = create_agent(
    model,
    tools=[search_code, search_issues, search_prs],
    system_prompt=(
        "You are a GitHub expert. Answer questions about code, "
        "API references, and implementation details by searching "
        "repositories, issues, and pull requests."
    ),
)

notion_agent = create_agent(
    model,
    tools=[search_notion, get_page],
    system_prompt=(
        "You are a Notion expert. Answer questions about internal "
        "processes, policies, and team documentation by searching "
        "the organization's Notion workspace."
    ),
)

slack_agent = create_agent(
    model,
    tools=[search_slack, get_thread],
    system_prompt=(
        "You are a Slack expert. Answer questions by searching "
        "relevant threads and discussions where team members have "
        "shared knowledge and solutions."
    ),
)


# Workflow nodes
def classify_query(state: RouterState) -> dict:
    """Classify query and determine which agents to invoke."""
    structured_llm = router_llm.with_structured_output(ClassificationResult)

    result = structured_llm.invoke([
        {
            "role": "system",
            "content": """Analyze this query and determine which knowledge bases to consult.
For each relevant source, generate a targeted sub-question optimized for that source.

Available sources:
- github: Code, API references, implementation details, issues, pull requests
- notion: Internal documentation, processes, policies, team wikis
- slack: Team discussions, informal knowledge sharing, recent conversations

Return ONLY the sources that are relevant to the query."""
        },
        {"role": "user", "content": state["query"]}
    ])

    return {"classifications": result.classifications}


def route_to_agents(state: RouterState) -> list[Send]:
    """Fan out to agents based on classifications."""
    return [
        Send(c["source"], {"query": c["query"]})
        for c in state["classifications"]
    ]


def query_github(state: AgentInput) -> dict:
    """Query the GitHub agent."""
    result = github_agent.invoke({
        "messages": [{"role": "user", "content": state["query"]}]
    })
    return {"results": [{"source": "github", "result": result["messages"][-1].content}]}


def query_notion(state: AgentInput) -> dict:
    """Query the Notion agent."""
    result = notion_agent.invoke({
        "messages": [{"role": "user", "content": state["query"]}]
    })
    return {"results": [{"source": "notion", "result": result["messages"][-1].content}]}


def query_slack(state: AgentInput) -> dict:
    """Query the Slack agent."""
    result = slack_agent.invoke({
        "messages": [{"role": "user", "content": state["query"]}]
    })
    return {"results": [{"source": "slack", "result": result["messages"][-1].content}]}


def synthesize_results(state: RouterState) -> dict:
    """Combine results from all agents into a coherent answer."""
    if not state["results"]:
        return {"final_answer": "No results found from any knowledge source."}

    formatted = [
        f"**From {r['source'].title()}:**\n{r['result']}"
        for r in state["results"]
    ]

    synthesis_response = router_llm.invoke([
        {
            "role": "system",
            "content": f"""Synthesize these search results to answer the original question: "{state['query']}"

- Combine information from multiple sources without redundancy
- Highlight the most relevant and actionable information
- Note any discrepancies between sources
- Keep the response concise and well-organized"""
        },
        {"role": "user", "content": "\n\n".join(formatted)}
    ])

    return {"final_answer": synthesis_response.content}


# Build workflow
workflow = (
    StateGraph(RouterState)
    .add_node("classify", classify_query)
    .add_node("github", query_github)
    .add_node("notion", query_notion)
    .add_node("slack", query_slack)
    .add_node("synthesize", synthesize_results)
    .add_edge(START, "classify")
    .add_conditional_edges("classify", route_to_agents, ["github", "notion", "slack"])
    .add_edge("github", "synthesize")
    .add_edge("notion", "synthesize")
    .add_edge("slack", "synthesize")
    .add_edge("synthesize", END)
    .compile()
)


if __name__ == "__main__":
    result = workflow.invoke({
        "query": "How do I authenticate API requests?"
    })

    print("Original query:", result["query"])
    print("\nClassifications:")
    for c in result["classifications"]:
        print(f"  {c['source']}: {c['query']}")
    print("\n" + "=" * 60 + "\n")
    print("Final Answer:")
    print(result["final_answer"])

Original query: How do I authenticate API requests?

Classifications:
  github: Check the API references and implementation details for authentication methods.
  notion: Look for internal documentation on API authentication processes and policies.
  slack: Search recent team discussions for informal knowledge sharing on API authentication.


Final Answer:
To authenticate API requests, you can use several methods, each with its own implementation details and best practices:

1. **OAuth2 Flow**: 
   - OAuth2 is a widely used authentication method that involves obtaining an access token through a series of steps, including user authorization. It allows for defining access levels and permissions using scopes. 
   - Relevant resources: GitHub Issue #89 and PR #178 discuss OAuth flow and scope updates.

2. **JWT (JSON Web Token)**:
   - JWTs are a popular method for securing APIs. They are compact, URL-safe tokens that contain claims and are used for authentication and information exchange.


In [5]:
import requests
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from markdownify import markdownify


ALLOWED_DOMAINS = ["https://langchain-ai.github.io/"]
LLMS_TXT = 'https://langchain-ai.github.io/langgraph/llms.txt'
# We will fetch the content of llms.txt, so this can
# be done ahead of time without requiring an LLM request.
llms_txt_content = requests.get(LLMS_TXT).text




@tool
def fetch_documentation(url: str) -> str:  
    """Fetch and convert documentation from a URL"""
    if not any(url.startswith(domain) for domain in ALLOWED_DOMAINS):
        return (
            "Error: URL not allowed. "
            f"Must start with one of: {', '.join(ALLOWED_DOMAINS)}"
        )
    response = requests.get(url, timeout=10.0)
    response.raise_for_status()
    return markdownify(response.text)

# System prompt for the agent
system_prompt = f"""
You are an expert Python developer and technical assistant.
Your primary role is to help users with questions about LangGraph and related tools.

Instructions:

1. If a user asks a question you're unsure about — or one that likely involves API usage,
   behavior, or configuration — you MUST use the `fetch_documentation` tool to consult the relevant docs.
2. When citing documentation, summarize clearly and include relevant context from the content.
3. Do not use any URLs outside of the allowed domain.
4. If a documentation fetch fails, tell the user and proceed with your best expert understanding.

You can access official documentation from the following approved sources:

{llms_txt_content}

You MUST consult the documentation to get up to date documentation
before answering a user's question about LangGraph.

Your answers should be clear, concise, and technically accurate.
"""

tools = [fetch_documentation]

model = init_llm(
    'anthropic--claude-3.5-sonnet', 
    temperature=0.1,
    max_tokens=12000
)

# model = init_chat_model("claude-sonnet-4-0", max_tokens=32_000)



agent = create_agent(
    model=model,
    tools=tools,  
    system_prompt=system_prompt,  
    name="Agentic RAG",
)

response = agent.invoke(
    {
        'messages': [
            HumanMessage(
                content=(
                    """
                    Write a short example of a langgraph agent using the prebuilt create react agent. the agent should be able to look up stock pricing information.
                    """
                )
            )
        ]
    }
)


# response =agent.invoke(
#     {
#         "messages":[
#             {
#                 "role": "user",
#                 "content": 
#                 """
#                 Write a short example of a langgraph agent using the prebuilt create react agent. the agent should be able to look up stock pricing information.
#                 """
#             }
#         ]
#     }
# )
# #print(response)




print(response['messages'][-1].content)

/home/user/projects/SAP_GenAI/env/lib/python3.13/site-packages/gen_ai_hub/proxy/langchain/amazon.py:201: UserWarning: WARNING! client_params is not default parameter.
                client_params was transferred to model_kwargs.
                Please confirm that client_params is what you intended.
  return ChatBedrock(


I apologize again for the difficulty in accessing the documentation. Since we're unable to fetch the most recent documentation, I'll provide an example based on my current knowledge of LangGraph and the create_react_agent. Please note that this example might not reflect the most up-to-date syntax or best practices, but it should give you a good starting point.

Here's a short example of a LangGraph agent using the prebuilt create_react_agent that can look up stock pricing information:

```python
from langchain.agents import create_react_agent
from langchain.chat_models import ChatOpenAI
from langchain.tools import Tool
from langgraph.graph import StateGraph, END

# Assuming you have a function to fetch stock prices
def get_stock_price(symbol: str) -> str:
    # This is a placeholder function. In a real scenario, you'd use an API or library to fetch actual stock prices.
    return f"The current price of {symbol} is $100.00"

# Define the tool
stock_price_tool = Tool(
    name="StockPric

In [69]:
def fetch_documentation(url: str) -> str:  
    """Fetch and convert documentation from a URL"""
    if not any(url.startswith(domain) for domain in ALLOWED_DOMAINS):
        return (
            "Error: URL not allowed. "
            f"Must start with one of: {', '.join(ALLOWED_DOMAINS)}"
        )
    response = requests.get(url, timeout=10.0)
    response.raise_for_status()
    return markdownify(response.text)

In [65]:
LLMS_TXT = 'https://langchain-ai.github.io/langgraph/llms.txt'

# We will fetch the content of llms.txt, so this can
# be done ahead of time without requiring an LLM request.
llms_txt_content = requests.get(LLMS_TXT).text


print(llms_txt_content)


# Guides

- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/index/): This page provides an overview of the LangGraph project, including its logo and essential scripts for functionality within MkDocs. It also includes a reference to the README.md file for detailed information about the project. The content is designed to be user-friendly and visually appealing.
- [LangGraph Quickstart Guide](https://langchain-ai.github.io/langgraph/agents/agents/): This quickstart guide provides step-by-step instructions for setting up and using LangGraph's prebuilt components to create agentic systems. It covers prerequisites, installation, agent creation, configuration of language models, and advanced features like memory and structured output. Ideal for developers looking to leverage LangGraph for building intelligent agents.
- [Getting Started with LangGraph: Building AI Agents](https://langchain-ai.github.io/langgraph/concepts/why-langgraph/): This page provides an overview of L

In [77]:
url = "https://langchain-ai.github.io/langgraph/tutorials/get-started/1-build-basic-chatbot/"
#url="https://www.duh.de/presse/pressemitteilungen/pressemitteilung/hvo100-noch-schmutziger-als-herkoemmlicher-diesel-abgasmessungen-der-deutschen-umwelthilfe-zerstoeren/"

fetch_documentation(url)

'Redirecting...\n\n\nRedirecting...'

## Multi-agent 

In [81]:

from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver  

@tool
def get_weather(latitude, longitude):
    """This is a publically available API that returns the weather for a given location."""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return data["current"]


from langchain.agents import create_agent

agent_weather1 = create_agent(
    model,
    tools = [get_weather],
    system_prompt=(
        "You are a helpful assistant help user to get the real time weather of a given city."
        "Use tool get_weather to fetch the current weather information."
    ),
    checkpointer=InMemorySaver(),    
)

In [82]:
messages=([
    {"messages": [{"role": "user", "content": "What is the weather in Shanghai?"}]},
    {"configurable": {"thread_id": "1"}},  
])

In [83]:
agent_weather1.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is the weather in Shanghai?"}
        ]
    },
    {
        "configurable": {"thread_id": "1"}
    },  
)

{'messages': [HumanMessage(content='What is the weather in Shanghai?', additional_kwargs={}, response_metadata={}, id='924b91d0-81f8-4b79-aaa8-acddfc854f46'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_f7qQKAjFiPUQR4lCr9YpOb8r', 'function': {'arguments': '{"latitude":31.2304,"longitude":121.4737}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 92, 'total_tokens': 117, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-CxRxrQncQGM6wByVh76F0c82NA223', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bb5fe-0908-7233-bb82-7eaf0c009e3f-0', tool_calls=[{'name': 'get_weather', 'args': {'latitud